# 03 — Feature Selection

**Dataset:** Global Coral Bleaching Database 1980–2020 (van Woesik & Kratochwill 2022)  
**Target variable:** `Bleaching_Category`

---

## Contents

1. Column inventory and categorisation
2. Drop of non-features
3. Drop of high-missingness columns
4. Multicollinearity analysis
5. Correlation with target
6. Feature set finalisation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy.stats import spearmanr

# Setting pandas to show all columns: the dataset has 60 columns and pandas truncates by default
pd.set_option('display.max_columns', None)

In [2]:
# Project colour palette: ocean themed 🌊
color_primary = 'teal'
color_secondary = 'coral'  
color_accent = 'turquoise'

## 1. Column inventory and categorization

Exploring the columns to verify which can be dropped and which are to be retained for multicollinearity analysis and feature selection.

In [3]:
# Importing the post-EDA data
df = pd.read_csv('../data/processed/bleaching_eda.csv', low_memory=False)
print(df.shape)
df.head()

(41361, 60)


,Site_ID,Sample_ID,Data_Source,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Reef_ID,Realm_Name,Ecoregion_Name,Country_Name,State_Island_Province_Name,City_Town_Name,Distance_to_Shore,Exposure,Turbidity,Cyclone_Frequency,Date_Day,Date_Month,Date_Year,Depth_m,Substrate_Name,Percent_Cover,Percent_Bleaching,ClimSST,Temperature_Kelvin,Temperature_Mean,Temperature_Minimum,Temperature_Maximum,Temperature_Kelvin_Standard_Deviation,Windspeed,SSTA,SSTA_Standard_Deviation,SSTA_Mean,SSTA_Minimum,SSTA_Maximum,SSTA_Frequency,SSTA_Frequency_Standard_Deviation,SSTA_FrequencyMax,SSTA_FrequencyMean,SSTA_DHW,SSTA_DHW_Standard_Deviation,SSTA_DHWMax,SSTA_DHWMean,TSA,TSA_Standard_Deviation,TSA_Minimum,TSA_Maximum,TSA_Mean,TSA_Frequency,TSA_Frequency_Standard_Deviation,TSA_FrequencyMax,TSA_FrequencyMean,TSA_DHW,TSA_DHW_Standard_Deviation,TSA_DHWMax,TSA_DHWMean,Date,Bleaching_Binary,Bleaching_Category,Bleaching_Severity_Ordinal
0,2501,10324336,Donner,23.163,-82.5260,Atlantic,NaN,Tropical Atlantic,Cuba and Cayman Islands,Cuba,Havana,Havana,8519.23,Exposed,0.0287,49.90,15,9,2005,10.00,NaN,NaN,50.2,301.61,302.05,300.67,296.72,304.69,1.60,8.0,-0.46,1.0,0.0,-3.56,2.24,0.0,3.13,17.00,3.0,0.00,1.63,7.88,0.98,-0.80,1.60,-6.12,1.83,-2.17,0.00,1.09,5.0,0.0,0.00,0.74,7.25,0.18,2005-09-15,1,moderate,2.0
1,3467,10324754,Donner,-17.575,-149.7833,Pacific,NaN,Eastern Indo-Pacific,Society Islands French Polynesia,French Polynesia,Society Islands,Moorea,1431.62,Exposed,0.0262,51.20,15,3,1991,14.00,NaN,NaN,50.7,262.15,303.30,300.73,297.58,305.01,1.12,2.0,1.29,1.0,0.0,-2.73,3.10,0.5,2.77,13.25,2.0,0.26,1.48,11.41,0.72,1.29,1.12,-4.42,3.00,-1.26,0.25,0.93,4.0,0.0,0.26,0.67,4.65,0.19,1991-03-15,1,moderate,2.0
2,1794,10323866,Donner,18.369,-64.5640,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United Kingdom,British Virgin Islands,Peter Island,182.33,Exposed,0.0429,61.52,15,1,2006,7.00,NaN,NaN,50.9,298.79,299.18,300.32,297.12,304.14,1.22,8.0,0.04,1.0,0.0,-2.92,2.83,16.0,4.52,23.00,3.0,0.00,2.45,16.24,1.26,-2.64,1.22,-4.69,2.31,-1.49,7.00,1.31,7.0,0.0,0.00,1.04,11.66,0.26,2006-01-15,1,moderate,2.0
3,8647,10328028,Donner,17.760,-64.5680,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,US Virgin Islands,St. Croix,313.13,Exposed,0.0424,65.39,15,4,2006,9.02,NaN,NaN,50.9,300.16,299.61,300.38,297.25,304.07,1.19,3.0,-0.07,1.0,0.0,-2.77,2.47,22.0,4.75,24.00,3.0,0.00,2.37,16.73,1.07,-2.27,1.19,-4.63,2.19,-1.49,3.00,0.94,4.0,0.0,0.00,0.75,5.64,0.20,2006-04-15,1,moderate,2.0
4,8648,10328029,Donner,17.769,-64.5830,Atlantic,NaN,Tropical Atlantic,Hispaniola Puerto Rico and Lesser Antilles,United States,US Virgin Islands,St. Croix,792.00,Exposed,0.0424,65.39,15,4,2006,12.50,NaN,NaN,50.9,300.15,299.70,300.38,296.63,303.76,1.18,3.0,0.00,1.0,0.0,-2.84,2.30,16.0,4.16,20.00,3.0,0.00,2.24,13.86,1.16,-2.19,1.18,-5.25,1.87,-1.50,3.00,1.33,5.0,0.0,0.00,0.92,6.89,0.25,2006-04-15,1,moderate,2.0


## 2. Drop of non-features

### Drop of identifier columns

The following columns are pure identifiers or geographic labels too granular to carry 
modelling signal. Latitude and longitude already identify each observation precisely.

- `Site_ID`, `Sample_ID`, `Reef_ID`: arbitrary numeric identifiers
- `City_Town_Name`, `State_Island_Province_Name`: too granular, redundant with coordinates
- `Country_Name`: redundant with higher-level geographic aggregations (`Ocean_Name`, `Realm_Name`)

In [4]:
# Dropping the identifier columns
cols_identifiers = [
    'Site_ID', 'Sample_ID', 'Reef_ID',
    'City_Town_Name', 'State_Island_Province_Name',
    'Country_Name']

df.drop(columns=cols_identifiers, inplace=True)
print(f"Columns remaining: {df.shape[1]}")

Columns remaining: 54


### Drop of target-derived columns

`Percent_Bleaching` is the continuous source variable from which `Bleaching_Category` 
was derived. `Bleaching_Binary` and `Bleaching_Severity_Ordinal` are alternative 
encodings of the same target. Keeping any of these would cause data leakage.

In [5]:
cols_target_derived = [
    'Percent_Bleaching',
    'Bleaching_Binary', 
    'Bleaching_Severity_Ordinal']

df.drop(columns=cols_target_derived, inplace=True)
print(f"Columns remaining: {df.shape[1]}")

Columns remaining: 51


### Date components

Investigating all Date columns prior to dropping any of them.

In [6]:
df['Date_Day'].value_counts()

Date_Day
15    7662
20    1270
22    1269
19    1256
21    1249
9     1248
10    1247
23    1245
13    1231
27    1227
24    1216
14    1206
17    1199
25    1194
12    1187
8     1181
7     1170
16    1161
29    1145
26    1143
11    1120
18    1117
6     1098
28    1094
30    1073
5     1010
1      962
4      887
2      855
3      853
31     586
Name: count, dtype: int64

### Drop of date components

`Date_Day` is too variable to carry ecological signal: it reflects survey scheduling, 
not bleaching dynamics. `Date` is the full datetime string, redundant with `Date_Year` 
and `Date_Month` which are already present as numeric columns.

The sanity check performed during data cleaning already confirmed that `Date` and `Date_Year` and `Date_Month` are consistent and do not carry different information.

In [7]:
cols_date = ['Date_Day', 'Date']

df.drop(columns=cols_date, inplace=True)
print(f"Columns remaining: {df.shape[1]}")

Columns remaining: 49


In [8]:
df.columns.tolist()

['Data_Source',
 'Latitude_Degrees',
 'Longitude_Degrees',
 'Ocean_Name',
 'Realm_Name',
 'Ecoregion_Name',
 'Distance_to_Shore',
 'Exposure',
 'Turbidity',
 'Cyclone_Frequency',
 'Date_Month',
 'Date_Year',
 'Depth_m',
 'Substrate_Name',
 'Percent_Cover',
 'ClimSST',
 'Temperature_Kelvin',
 'Temperature_Mean',
 'Temperature_Minimum',
 'Temperature_Maximum',
 'Temperature_Kelvin_Standard_Deviation',
 'Windspeed',
 'SSTA',
 'SSTA_Standard_Deviation',
 'SSTA_Mean',
 'SSTA_Minimum',
 'SSTA_Maximum',
 'SSTA_Frequency',
 'SSTA_Frequency_Standard_Deviation',
 'SSTA_FrequencyMax',
 'SSTA_FrequencyMean',
 'SSTA_DHW',
 'SSTA_DHW_Standard_Deviation',
 'SSTA_DHWMax',
 'SSTA_DHWMean',
 'TSA',
 'TSA_Standard_Deviation',
 'TSA_Minimum',
 'TSA_Maximum',
 'TSA_Mean',
 'TSA_Frequency',
 'TSA_Frequency_Standard_Deviation',
 'TSA_FrequencyMax',
 'TSA_FrequencyMean',
 'TSA_DHW',
 'TSA_DHW_Standard_Deviation',
 'TSA_DHWMax',
 'TSA_DHWMean',
 'Bleaching_Category']

### Further inspection of `Percent_Cover` and `Cyclone_Frequency`

Before proceeding with multicollinearity analysis, two columns require closer inspection: 
`Percent_Cover` and `Cyclone_Frequency`. Their ecological meaning and data quality need 
to be assessed before deciding whether to retain them as features.

In [9]:
# Investigating the 'Percent_Coverage' column
print(df['Percent_Cover'].describe().T)
print(df['Percent_Cover'].isna().sum())

count    28906.000000
mean        19.418280
std         20.850784
min          0.000000
25%          0.620000
50%         12.500000
75%         33.120000
max        100.000000
Name: Percent_Cover, dtype: float64
12455


In [10]:
# Investigating the 'Cyclone_Frequency' column
print(df['Cyclone_Frequency'].describe().T)
print(df['Cyclone_Frequency'].dtype)
print(df['Cyclone_Frequency'].isna().sum())

count    41361.000000
mean        52.159650
std          7.589593
min         18.310000
25%         47.940000
50%         50.920000
75%         55.730000
max        105.800000
Name: Cyclone_Frequency, dtype: float64
float64
0


`Percent_Cover` indicates the coralline coverage of the site. The descriptive statistics 
suggest a highly right-skewed distribution: the standard deviation is higher than the mean, 
indicating that most sites have low coral coverage while few have high coverage. 
The feature may carry ecological significance, but the high missingness (~30%) will be 
evaluated at a later stage of the feature selection.

`Cyclone_Frequency` has no missing values and an approximately normal distribution. 
This feature may be linked to thermal anomalies and will be further explored during 
the multicollinearity analysis.

## 3. Drop of high-missingness columns

In [11]:
# Investigating the percentage missingness of data in all remaning columns
missing_count = df.isnull().mean()
missing_pct = missing_count * 100
missing_sorted = missing_pct.sort_values(ascending=False)
missing_only = missing_sorted[missing_sorted > 0]
print(missing_only.round(2))

Substrate_Name                           30.63
Percent_Cover                            30.11
Bleaching_Category                       16.55
Depth_m                                   4.35
SSTA_Minimum                              0.43
SSTA                                      0.36
SSTA_Frequency                            0.36
Temperature_Kelvin                        0.36
TSA_Frequency                             0.36
SSTA_DHW                                  0.36
TSA                                       0.36
TSA_DHW                                   0.36
Temperature_Kelvin_Standard_Deviation     0.32
SSTA_Maximum                              0.32
SSTA_Standard_Deviation                   0.32
SSTA_Mean                                 0.32
SSTA_Frequency_Standard_Deviation         0.32
TSA_Mean                                  0.32
SSTA_DHW_Standard_Deviation               0.32
SSTA_FrequencyMax                         0.32
SSTA_FrequencyMean                        0.32
TSA_Standard_

In [12]:
print(f"Bleaching_Category NaN count: {df['Bleaching_Category'].isna().sum()}")
print(f"Total rows: {len(df)}")
print(f"Rows with valid target: {df['Bleaching_Category'].notna().sum()}")

Bleaching_Category NaN count: 6846
Total rows: 41361
Rows with valid target: 34515


`Bleaching_Category` has 6,846 missing values (16.55% of 41,361 rows).


These are original missing values from the raw dataset, not introduced during cleaning, 
as verified in Notebook 01. These rows have valid environmental features but no target 
value, and will be dropped immediately before modelling.

The remaining 34,515 rows have a valid target and will form the modelling dataset.

In [13]:
# Checking for correlation between 'Percent_Cover' and our target variable
severity_map = {'none': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
temp_ordinal = df['Bleaching_Category'].map(severity_map)

valid = df[['Percent_Cover']].copy()
valid['target'] = temp_ordinal
valid = valid.dropna()

corr, pval = spearmanr(valid['Percent_Cover'], valid['target'])
print(f"Spearman r: {corr:.3f}, p-value: {pval:.4f}, n: {len(valid)}")

Spearman r: 0.041, p-value: 0.0000, n: 22673


### Drop of `Percent_Cover`

The Spearman correlation between `Percent_Cover` and `Bleaching_Category` is r = 0.041,
negligible. The p-value is 0.0000, but this is an artefact of the large sample size 
(n = 22,673): with tens of thousands of observations, even trivial correlations become 
statistically significant. The effect size is what matters here, and it is negligible.

Given the high missingness (30.11%) and the lack of predictive signal, imputing missing 
values would introduce noise rather than information. The column is therefore dropped.

In [14]:
# Dropping 'Percent_Cover' column
df.drop(columns=['Percent_Cover'], inplace=True)
print(f"Columns remaining: {df.shape[1]}")

Columns remaining: 48
